In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import sys
sys.path.append('/home/jovyan/work/base_demo')
import base_tool

In [14]:
import pandas as pd
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 65)

数据介绍

bid_book_begin 集合竞价后的完整委托买入订单簿

ask_book_begin 集合竞价后的完整委托卖出订单簿

snap_list 连续竞价阶段的1s快照
    time_hms  时分秒字符串
    time_mark 毫秒级时间戳
    price_open 快照内首个成交价(无成交时为0.0)
    price_low  快照内最低成交价(无成交时为0.0)
    price_high 快照内最高成交价(无成交时为0.0)
    price_last 当日内最新成交价
     buy_trade 主动买入成交
    sell_trade 主动卖出成交
    bid_insert 委托买入挂单
    ask_insert 委托卖出挂单
    bid_cancel 委托买入撤单
    ask_cancel 委托卖出撤单

In [15]:
instrument_id = '511520'
trade_ymd = '20260319'
day = 5

In [16]:
param_dict = {
    'name' : 'bollingger_bands',
    'instrument_id' : instrument_id,
    'trade_ymd' : trade_ymd,

    'ma_window': 300,
    'std_window': 300,
    'beta': 2
}

In [17]:
%%time
import os
import numpy as np
from collections import deque
import math

class StrategyDemo():
    def __init__(self, param_dict={}) -> None:
        # self.model = joblib.load(file)
        
        self.__dict__.update(param_dict)
        data_file = f'/home/jovyan/work/backtest_result/{self.instrument_id}_{self.trade_ymd}_{self.name}.pkl' 
        if os.path.exists(data_file):
            os.remove(data_file)

        self.position_last = 0

        self.max_len = max(self.ma_window,self.std_window)
        self.price_list = deque(maxlen=self.max_len)

        self.prev_signal = 0
        return

    def on_snap(self, snap:dict) -> None:
        price = snap['price_last']

        if price == 0.0 or price == None:
            return
        
        self.price_list.append(price)

        if(len(self.price_list) < self.max_len):
            self.position_last = 0
            self.prev_signal = 0
            return

        window_data = list(self.price_list)[-self.max_len:]

        ma_slice = window_data[-self.ma_window:]
        ma = sum(ma_slice) / self.ma_window

        std_slice = window_data[-self.std_window:]
        mean_of_squares = sum(x * x for x in std_slice) / self.std_window
        mean = sum(std_slice) / self.std_window
        square_of_mean = mean * mean
        variance = max(0, mean_of_squares - square_of_mean)
        std = math.sqrt(variance)

        if std > ma * 0.5 or std < 0.000001:
            print(f"⚠️ 异常波动警告: price={price}, ma={ma}, std={std}")
            return
        
        upper_band = ma + self.beta * std
        lower_band = ma - self.beta * std

        signal = self.prev_signal 

        if self.position_last == 0:
            if price < lower_band:
                signal = 1  
            elif price > upper_band:
                signal = -1
            
        elif self.position_last == 1: # 持有多单
            # 平仓条件：价格回归到均线之上，或者价格触及上轨（反向信号）
            if price >= ma or price >= upper_band:
                signal = 0 
            # 否则保持持有 (signal = 1)

        elif self.position_last == -1: # 持有空单
            # 平仓条件：价格回归到均线之下，或者价格触及下轨（反向信号）
            if price <= ma or price <= lower_band:
                signal = 0
            # 否则保持持有 (signal = -1)

        # 5. 更新状态
        self.position_last = signal
        self.prev_signal = signal
        
        return

CPU times: user 16 μs, sys: 0 ns, total: 16 μs
Wall time: 18.6 μs


In [18]:
%%time
def backtest_demo(instrument_id, trade_ymd, strategy_name):
    strategy = StrategyDemo(param_dict)
    snap_list = base_tool.snap_list_load(instrument_id, trade_ymd)
    position_dict = {}
    for snap in snap_list[:]:
        strategy.on_snap(snap)
        position_dict[snap['time_mark']] = strategy.position_last
    profit = base_tool.backtest_quick(instrument_id, trade_ymd, strategy_name, position_dict)
    return profit
backtest_demo(instrument_id, trade_ymd, param_dict['name'])

/home/jovyan/work/backtest_result/511520_20260319_bollingger_bands.pkl 生成中
CPU times: user 252 ms, sys: 8.99 ms, total: 261 ms
Wall time: 761 ms


,order_time,order_price,total,trade,cancel,hold,profit_last,profits,maxdd,MAR,pper
0,2026-03-19 09:35:11,116.032,1,1,0,-100,0.0,0.0,1.0,0.00,0.00
1,2026-03-19 09:48:09,116.074,2,2,0,0,-4.2,-4.2,4.2,-1.00,-2.10
2,2026-03-19 09:50:46,116.075,3,3,0,100,0.0,-4.2,4.2,-1.00,-1.40
3,2026-03-19 09:50:48,116.070,4,4,0,0,-0.5,-4.7,4.7,-1.00,-1.18
4,2026-03-19 09:50:49,116.074,5,5,0,100,0.0,-4.7,4.7,-1.00,-0.94
...,...,...,...,...,...,...,...,...,...,...,...
95,2026-03-19 14:46:59,116.185,99,96,3,0,0.0,-28.0,28.0,-1.00,-0.29
96,2026-03-19 14:47:31,116.180,100,97,3,100,0.0,-28.0,28.0,-1.00,-0.29
97,2026-03-19 14:53:22,116.168,101,98,3,0,-1.2,-29.2,29.2,-1.00,-0.30
98,2026-03-19 14:54:05,116.173,102,99,3,-100,0.0,-29.2,29.2,-1.00,-0.29


In [19]:
import os
import sys
current_notebook_path = os.path.abspath('%pwd') 
current_dir = os.path.dirname(current_notebook_path)
parent_dir = os.path.dirname(current_dir)
utils_path = os.path.join(parent_dir, 'tools')
sys.path.append(utils_path)

In [20]:
from backtesting import backtest_multi_days
backtest_df = backtest_multi_days(instrument_id,'20260202','20260320',StrategyDemo,param_dict)


/home/jovyan/work/backtest_result/511520_20260202_bollingger_bands.pkl 生成中
日期 20260202 回测完成，当日盈亏: -12.50
/home/jovyan/work/backtest_result/511520_20260203_bollingger_bands.pkl 生成中
日期 20260203 回测完成，当日盈亏: -18.20
/home/jovyan/work/backtest_result/511520_20260204_bollingger_bands.pkl 生成中
无交易
日期 20260204 无交易记录
/home/jovyan/work/backtest_result/511520_20260205_bollingger_bands.pkl 生成中
日期 20260205 回测完成，当日盈亏: -22.20
/home/jovyan/work/backtest_result/511520_20260206_bollingger_bands.pkl 生成中
日期 20260206 回测完成，当日盈亏: -12.00

instrument_id 511090
20260202
20260203
20260204
20260205
20260206
20260209
20260210
20260211
20260212
20260213
20260224
20260225
20260226
20260227
20260302
20260303
20260304
20260305
20260306
20260309
20260310
20260311
20260312
20260313
20260316
20260317
20260318
20260319
20260320

instrument_id 511520
20260202
20260203
20260204
20260205
20260206
20260209
20260210
20260211
20260212
20260213
20260224
20260225
20260226
20260227
20260302
20260303
20260304
20260305
20260306
2026030

In [21]:
print(backtest_df)

   trade_ymd          order_time  order_price  total  trade  cancel  hold  \
0   20260202 2026-02-02 14:54:51      115.317    107    104       3     0   
1   20260203 2026-02-03 14:55:03      115.328     66     64       2     0   
2   20260205 2026-02-05 14:54:50      115.398     84     82       2     0   
3   20260206 2026-02-06 14:55:00      115.531     74     71       3     0   
4   20260210 2026-02-10 14:52:52      115.745    103    100       3     0   
..       ...                 ...          ...    ...    ...     ...   ...   
21  20260316 2026-03-16 14:55:00      115.794    102    100       2     0   
22  20260317 2026-03-17 14:55:00      115.895    130    126       4     0   
23  20260318 2026-03-18 14:54:45      116.029     99     94       5     0   
24  20260319 2026-03-19 14:55:00      116.170    103    100       3     0   
25  20260320 2026-03-20 14:55:00      116.062    112    112       0     0   

    profit_last  profits  maxdd   MAR  pper  
0          -0.1    -12.5   12

In [22]:
from backtesting import backtest_summary
summary = backtest_summary(backtest_df)
print(summary)

{'交易天数': 26, '累计盈亏': np.float64(-449.9), '最大单日盈利': -8.0, '最大单日亏损': -28.9, '盈利天数': 0, '亏损天数': 26, '平盘天数': 0, '胜率(%)': np.float64(0.0), '日均盈亏': np.float64(-17.3), '盈亏比': np.float64(0.0)}
